# GovernanceFund — 백테스트 시나리오

**목적**: 투표 알고리즘 + 실제 가격 데이터로 펀드 수익률 시뮬레이션  
**시나리오**: 모멘텀 / 역추세 / 랜덤 / 완벽예측 투표 비교  
**포함**: 레버리지, 롱/숏, 수수료, 청산 리스크

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
import requests
from datetime import datetime, timezone, timedelta
import warnings
warnings.filterwarnings('ignore')

# Windows 한국어 폰트 설정
_korean_fonts = ['Malgun Gothic', 'NanumGothic', 'AppleGothic', 'DejaVu Sans']
for _f in _korean_fonts:
    if any(_f.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = _f
        print(f'✅ 폰트 설정: {_f}')
        break
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (16, 7)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── 핵심 설정 ──────────────────────────────────────────────────
VOLATILITY = {'BTC': 0.030, 'ETH': 0.040, 'SOL': 0.055, 'HYPE': 0.070}
COINS = list(VOLATILITY.keys())

# 펀드 프로파일
#   T_CONVERGE = 목표 비중의 90%까지 도달하는 데 걸리는 '달력 일수'
#   → alpha는 투표주기(REBALANCE_EVERY)에서 자동 계산 (적응형)
#   공격적: 7일이면 거의 반영(빠른 반응) / 보수적: 21일에 걸쳐 반영(느린 반응)
FUND_PROFILES = {
    'aggressive':  {'T_CONVERGE': 7,  'MAX_WEIGHT': 80, 'FUND_LEVERAGE': 5, 'color': '#e74c3c'},
    'conservative':{'T_CONVERGE': 21, 'MAX_WEIGHT': 60, 'FUND_LEVERAGE': 2, 'color': '#2ecc71'},
}

TAKER_FEE = 0.00045
MAKER_FEE = 0.00010
REBALANCE_FEE = MAKER_FEE * 2


def adaptive_alpha(period_days, T_converge_days, leftover=0.1):
    """
    투표주기 적응형 EMA 계수.
    어떤 투표주기를 쓰든 'T_converge일 안에 (1-leftover)=90% 수렴'을 보장.
      alpha = 1 - leftover ^ (투표주기 / 목표수렴기간)
    - 주기가 길수록(주간) alpha↑ → 한 번에 확 반영
    - 주기가 짧을수록(일간) alpha↓ → 살살 반영하되 자주 누적
    결과적으로 '달력 기준' 반응속도는 동일.
    """
    period_days = max(period_days, 1e-9)
    return 1 - leftover ** (period_days / T_converge_days)


def simulate_votes(participants, coins):
    """예치금 가중평균으로 코인별 score(-1~+1) 산출"""
    total = sum(p['deposit'] for p in participants)
    results = {}
    for coin in coins:
        ws = sum(p['votes'].get(coin, 0) * p['deposit'] / total for p in participants)
        results[coin] = {'score': ws / 2}
    return results


print('✅ 설정 완료')
print('\n[적응형 alpha 미리보기]  (수렴기간 T별로 투표주기→alpha)')
for tname, prof in FUND_PROFILES.items():
    T = prof['T_CONVERGE']
    row = f"  {tname:<12} T={T:>2}일 →"
    for period, plabel in [(7,'주간'),(1,'일간')]:
        row += f"  {plabel} alpha={adaptive_alpha(period, T):.3f}"
    print(row)

✅ 폰트 설정: Malgun Gothic
✅ 설정 완료

[적응형 alpha 미리보기]  (수렴기간 T별로 투표주기→alpha)
  aggressive   T= 7일 →  주간 alpha=0.900  일간 alpha=0.280
  conservative T=21일 →  주간 alpha=0.536  일간 alpha=0.104


## 1. 가격 데이터 수집 (HL API)

In [2]:
HL_API = 'https://api.hyperliquid.xyz/info'

def fetch_candles(coin, days=180, interval='1d'):
    """HL candle API로 일간 OHLCV 수집"""
    end_ms   = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_ms = end_ms - days * 86400 * 1000

    # interval 값: '1m','5m','15m','1h','4h','1d'
    res = requests.post(HL_API, json={
        'type': 'candleSnapshot',
        'req': {'coin': coin, 'interval': interval,
                'startTime': start_ms, 'endTime': end_ms}
    }, timeout=15)

    if res.status_code != 200 or not res.json():
        print(f'  {coin}: API 오류 또는 데이터 없음')
        return None

    df = pd.DataFrame(res.json())
    df['time'] = pd.to_datetime(df['t'], unit='ms', utc=True)
    df = df.rename(columns={'o':'open','h':'high','l':'low','c':'close','v':'volume'})
    df = df[['time','open','high','low','close','volume']].set_index('time')
    df = df.astype({'open':float,'high':float,'low':float,'close':float,'volume':float})
    return df.sort_index()


print('📡 가격 데이터 수집 중...')
price_data = {}
for coin in COINS:
    df = fetch_candles(coin, days=180)
    if df is not None and len(df) > 10:
        price_data[coin] = df
        print(f'  {coin}: {len(df)}일치 수집 ({df.index[0].date()} ~ {df.index[-1].date()})')
    else:
        print(f'  {coin}: 수집 실패 → 합성 데이터로 대체')

# 수집 실패한 코인은 합성 데이터로 대체
np.random.seed(99)
if price_data:
    ref_dates = next(iter(price_data.values())).index
else:
    ref_dates = pd.date_range(end=datetime.now(timezone.utc), periods=180, freq='D', tz='UTC')

for coin in COINS:
    if coin not in price_data:
        vol = VOLATILITY[coin]
        returns = np.random.normal(0.001, vol, len(ref_dates))
        prices = 1000 * np.cumprod(1 + returns)
        price_data[coin] = pd.DataFrame({'close': prices}, index=ref_dates)
        print(f'  {coin}: 합성 데이터 생성')

# 공통 날짜로 정렬
common_idx = ref_dates
closes = pd.DataFrame({c: price_data[c]['close'].reindex(common_idx, method='ffill')
                       for c in COINS})
returns = closes.pct_change().fillna(0)
print(f'\n📊 공통 데이터: {len(closes)}일 × {len(COINS)}코인')
print(closes.tail(3).round(2))

📡 가격 데이터 수집 중...
  BTC: 181일치 수집 (2025-12-12 ~ 2026-06-10)


  ETH: 181일치 수집 (2025-12-12 ~ 2026-06-10)
  SOL: 181일치 수집 (2025-12-12 ~ 2026-06-10)


  HYPE: 181일치 수집 (2025-12-12 ~ 2026-06-10)

📊 공통 데이터: 181일 × 4코인
                               BTC     ETH    SOL   HYPE
time                                                    
2026-06-08 00:00:00+00:00  63058.0  1688.8  66.77  63.84
2026-06-09 00:00:00+00:00  61695.0  1638.5  64.92  57.76
2026-06-10 00:00:00+00:00  61533.0  1633.7  63.85  55.62


## 2. 투표 시나리오 정의

| 시나리오 | 설명 |
|----------|------|
| Momentum | 최근 2주 수익률 방향으로 투표 (추세 추종) |
| Contrarian | 최근 2주 수익률 반대로 투표 (역추세) |
| Random | 완전 랜덤 투표 |
| Perfect | 다음 2주 수익률을 미리 알고 투표 (상한선) |
| BuyHold | 투표 없이 동일 비중 유지 (벤치마크) |

In [3]:
REBALANCE_EVERY = 7   # 7일마다 리밸런싱 (주간)
LOOKBACK = 14         # 모멘텀 기준 과거 14일

def score_to_vote(score_float):
    """연속 score → 5단계 이산 투표 (-2~+2)"""
    if score_float >  0.5: return  2
    if score_float >  0.1: return  1
    if score_float < -0.5: return -2
    if score_float < -0.1: return -1
    return 0

def generate_votes_momentum(returns_window, participants):
    """최근 수익률 방향으로 투표"""
    for p in participants:
        for coin in COINS:
            ret = returns_window[coin].sum() if coin in returns_window else 0
            p['votes'][coin] = score_to_vote(ret / 0.1)  # 0.1 기준 정규화
    return participants

def generate_votes_contrarian(returns_window, participants):
    """최근 수익률 반대로 투표"""
    for p in participants:
        for coin in COINS:
            ret = returns_window[coin].sum() if coin in returns_window else 0
            p['votes'][coin] = score_to_vote(-ret / 0.1)
    return participants

def generate_votes_random(participants, seed=None):
    """완전 랜덤"""
    rng = np.random.default_rng(seed)
    for p in participants:
        for coin in COINS:
            p['votes'][coin] = int(rng.choice([-2,-1,0,1,2], p=[0.1,0.2,0.4,0.2,0.1]))
    return participants

def generate_votes_perfect(returns_forward, participants):
    """미래 수익률 알고 투표 (이론적 상한)"""
    for p in participants:
        for coin in COINS:
            ret = returns_forward[coin].sum() if coin in returns_forward else 0
            p['votes'][coin] = score_to_vote(ret / 0.1)
    return participants

print('✅ 투표 시나리오 함수 정의 완료')

✅ 투표 시나리오 함수 정의 완료


## 3. 백테스트 엔진

In [4]:
# ── 투표 → 목표 비중 (target 방식) ─────────────────────────────────
def votes_to_target(vote_result):
    """
    집단 투표 결과 → 목표 비중 (부호 있음)
    score가 곧바로 목표 방향/크기를 결정 (증분 누적이 아님)
      score = +1 → 풀 롱 목표 / score = -1 → 풀 숏 목표
    변동성 높은 코인은 같은 score라도 작은 비중 (vol_factor)
    정규화: sum(abs) = 100  (gross exposure 100%)
    신호 없음(all hold) → None → 현 비중 유지
    """
    raw = {}
    for c in COINS:
        score = vote_result[c]['score']
        vol_factor = 0.03 / VOLATILITY.get(c, 0.05)
        raw[c] = score * vol_factor

    total_abs = sum(abs(v) for v in raw.values())
    if total_abs < 1e-9:
        return None
    return {c: raw[c] / total_abs * 100 for c in COINS}


# ── 백테스트 엔진 (target + 적응형 alpha) ──────────────────────────
def run_backtest(scenario_name, vote_generator, returns_df, profile_name,
                 initial_tvl=100_000, rebalance_every=None):
    profile = FUND_PROFILES[profile_name]
    period  = rebalance_every if rebalance_every is not None else REBALANCE_EVERY

    # 적응형 alpha: 투표주기와 프로파일의 목표수렴기간으로 자동 계산
    alpha = adaptive_alpha(period, profile['T_CONVERGE'])

    participants = [
        {'name': 'A', 'deposit': 500, 'votes': {}},
        {'name': 'B', 'deposit': 300, 'votes': {}},
        {'name': 'C', 'deposit': 200, 'votes': {}},
    ]

    lev    = profile['FUND_LEVERAGE']
    cap    = profile['MAX_WEIGHT']
    equity = initial_tvl
    weights = {coin: 100.0 / len(COINS) for coin in COINS}  # 초기 균등 롱

    history    = []
    liquidated = False

    for day_idx, date in enumerate(returns_df.index):

        # ── 리밸런싱 (period일마다) ─────────────────────────────────
        if day_idx % period == 0 and day_idx > 0:
            past_slice   = returns_df.iloc[max(0, day_idx - LOOKBACK):day_idx]
            future_slice = returns_df.iloc[day_idx:day_idx + period]

            participants = vote_generator(day_idx, participants,
                                          past_slice, future_slice)
            vote_result = simulate_votes(participants, COINS)
            target = votes_to_target(vote_result)

            if target is not None:
                new_weights = {
                    c: float(np.clip(alpha * target[c] + (1 - alpha) * weights[c],
                                     -cap, cap))
                    for c in COINS
                }
                turnover = sum(abs(new_weights[c] - weights[c]) for c in COINS) / 100
                equity  -= turnover * equity * lev * REBALANCE_FEE
                weights  = new_weights

        # ── 일간 PnL (부호가 롱/숏 방향 자동 처리) ──────────────────
        daily_pnl = sum(
            (weights[coin] / 100) * equity * lev * returns_df.loc[date, coin]
            for coin in COINS
        )
        equity += daily_pnl

        if equity < initial_tvl * 0.1 and not liquidated:
            liquidated = True
            print(f'  ⚠️  {scenario_name}/{profile_name}: Day {day_idx} 청산 발생')

        history.append({
            'date':   date,
            'equity': max(equity, 0),
            **{f'w_{c}': weights[c] for c in COINS}
        })

    df = pd.DataFrame(history).set_index('date')
    df['return']     = df['equity'].pct_change().fillna(0)
    df['cum_return'] = (df['equity'] / initial_tvl - 1) * 100
    df['drawdown']   = (df['equity'] / df['equity'].cummax() - 1) * 100
    df['scenario']   = scenario_name
    df['profile']    = profile_name
    df['liquidated'] = liquidated
    return df


def make_generator(scenario):
    def gen(day_idx, participants, past, future):
        if scenario == 'momentum':   return generate_votes_momentum(past, participants)
        if scenario == 'contrarian': return generate_votes_contrarian(past, participants)
        if scenario == 'perfect':    return generate_votes_perfect(future, participants)
        return generate_votes_random(participants, seed=day_idx)
    return gen

print('✅ 백테스트 엔진 (target + 적응형 alpha)')
print()
print('  • votes_to_target: score → 목표 포지션 (풀롱~풀숏)')
print('  • alpha = adaptive_alpha(투표주기, 프로파일 T_CONVERGE)')
print(f'    현재 주간(7일) 기준:')
for pn, pf in FUND_PROFILES.items():
    print(f'      {pn:<12} alpha = {adaptive_alpha(7, pf["T_CONVERGE"]):.3f}  (T={pf["T_CONVERGE"]}일)')
print('  • 검증 기준: Perfect > 나머지 > Buy&Hold 순이어야 엔진 정상')

✅ 백테스트 엔진 (target + 적응형 alpha)

  • votes_to_target: score → 목표 포지션 (풀롱~풀숏)
  • alpha = adaptive_alpha(투표주기, 프로파일 T_CONVERGE)
    현재 주간(7일) 기준:
      aggressive   alpha = 0.900  (T=7일)
      conservative alpha = 0.536  (T=21일)
  • 검증 기준: Perfect > 나머지 > Buy&Hold 순이어야 엔진 정상


In [5]:
SCENARIOS = ['momentum', 'contrarian', 'random', 'perfect']
SCENARIO_LABELS = {
    'momentum':   '모멘텀 (추세 추종)',
    'contrarian': '역추세',
    'random':     '랜덤 투표',
    'perfect':    '완벽 예측 (상한선)',
}

# Buy & Hold 벤치마크
buyhold_equity = (1 + returns.mean(axis=1)).cumprod() * 100_000
buyhold_cum    = (buyhold_equity / 100_000 - 1) * 100

results = {}
print('🚀 백테스트 실행 중...')
for profile_name in FUND_PROFILES:
    results[profile_name] = {}
    for scenario in SCENARIOS:
        df = run_backtest(
            scenario_name=scenario,
            vote_generator=make_generator(scenario),
            returns_df=returns,
            profile_name=profile_name,
        )
        results[profile_name][scenario] = df
        final_ret = df['cum_return'].iloc[-1]
        max_dd    = df['drawdown'].min()
        liq       = '⚠️ 청산' if df['liquidated'].iloc[-1] else '정상'
        print(f"  [{profile_name[:4]}] {SCENARIO_LABELS[scenario]:<20} "
              f"누적수익: {final_ret:>+7.1f}%  최대손실: {max_dd:>6.1f}%  {liq}")
    print()

🚀 백테스트 실행 중...
  [aggr] 모멘텀 (추세 추종)          누적수익:   -48.4%  최대손실:  -84.8%  정상
  ⚠️  contrarian/aggressive: Day 75 청산 발생
  [aggr] 역추세                  누적수익:   -97.2%  최대손실:  -97.6%  ⚠️ 청산
  [aggr] 랜덤 투표                누적수익:   +26.7%  최대손실:  -55.4%  정상
  [aggr] 완벽 예측 (상한선)          누적수익: +75638.1%  최대손실:  -51.2%  정상

  [cons] 모멘텀 (추세 추종)          누적수익:   -11.4%  최대손실:  -38.1%  정상
  [cons] 역추세                  누적수익:   -37.7%  최대손실:  -42.6%  정상
  [cons] 랜덤 투표                누적수익:    -3.7%  최대손실:  -23.7%  정상


  [cons] 완벽 예측 (상한선)          누적수익:  +550.3%  최대손실:  -23.7%  정상



## 4. 시각화

In [6]:
# ── 수익률 곡선 비교 ──────────────────────────────────────────────
scenario_colors = {
    'momentum':   '#3498db',
    'contrarian': '#e74c3c',
    'random':     '#95a5a6',
    'perfect':    '#f1c40f',
}

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('GovernanceFund 백테스트 — 시나리오별 수익률', fontsize=15, fontweight='bold')

for p_idx, (profile_name, profile) in enumerate(FUND_PROFILES.items()):
    ax_ret = axes[p_idx][0]
    ax_dd  = axes[p_idx][1]

    # 수익률 곡선
    ax_ret.plot(buyhold_cum.index, buyhold_cum.values,
                color='black', linestyle='--', linewidth=1.5,
                label='Buy & Hold (균등)', alpha=0.7)

    for scenario in SCENARIOS:
        df = results[profile_name][scenario]
        ax_ret.plot(df.index, df['cum_return'],
                    color=scenario_colors[scenario], linewidth=2,
                    label=SCENARIO_LABELS[scenario])

    ax_ret.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax_ret.set_title(f'{profile_name.upper()} — 누적 수익률 (레버리지 {profile["FUND_LEVERAGE"]}x)', fontsize=12)
    ax_ret.set_ylabel('누적 수익률 (%)')
    ax_ret.legend(fontsize=9)

    # 최대 낙폭
    for scenario in SCENARIOS:
        df = results[profile_name][scenario]
        ax_dd.fill_between(df.index, df['drawdown'], 0,
                           color=scenario_colors[scenario], alpha=0.3,
                           label=SCENARIO_LABELS[scenario])
        ax_dd.plot(df.index, df['drawdown'],
                   color=scenario_colors[scenario], linewidth=1)

    ax_dd.set_title(f'{profile_name.upper()} — 최대 낙폭 (Drawdown)', fontsize=12)
    ax_dd.set_ylabel('낙폭 (%)')
    ax_dd.legend(fontsize=9)

plt.tight_layout()
plt.savefig('backtest_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ backtest_equity_curves.png 저장')

✅ backtest_equity_curves.png 저장


In [7]:
# ── 성과 지표 비교표 ────────────────────────────────────────────
def calc_metrics(df, initial=100_000):
    r = df['return']
    ann_ret  = df['cum_return'].iloc[-1] * 365 / len(df)
    ann_vol  = r.std() * np.sqrt(365) * 100
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else 0
    max_dd   = df['drawdown'].min()
    calmar   = ann_ret / abs(max_dd) if max_dd < 0 else np.inf
    win_rate = (r > 0).mean() * 100
    return {
        '연환산 수익률(%)': round(ann_ret, 1),
        '연환산 변동성(%)': round(ann_vol, 1),
        'Sharpe':          round(sharpe, 2),
        '최대낙폭(%)':      round(max_dd, 1),
        'Calmar':          round(calmar, 2),
        '승률(%)':          round(win_rate, 1),
    }

print('\n📊 성과 지표 요약')
print('=' * 90)

for profile_name in FUND_PROFILES:
    rows = []
    idx  = []
    for scenario in SCENARIOS:
        rows.append(calc_metrics(results[profile_name][scenario]))
        idx.append(SCENARIO_LABELS[scenario])

    # Buy & Hold
    bh_df = pd.DataFrame({'return': buyhold_equity.pct_change().fillna(0),
                          'cum_return': buyhold_cum,
                          'drawdown': (buyhold_equity / buyhold_equity.cummax() - 1) * 100})
    rows.append(calc_metrics(bh_df))
    idx.append('Buy & Hold (벤치마크)')

    metrics_df = pd.DataFrame(rows, index=idx)
    print(f'\n[{profile_name.upper()} | 레버리지 {FUND_PROFILES[profile_name]["FUND_LEVERAGE"]}x]')
    print(metrics_df.to_string())
    print()


📊 성과 지표 요약

[AGGRESSIVE | 레버리지 5x]
                   연환산 수익률(%)  연환산 변동성(%)  Sharpe  최대낙폭(%)   Calmar  승률(%)
모멘텀 (추세 추종)             -97.7       241.2   -0.40    -84.8    -1.15   50.3
역추세                    -196.1       242.0   -0.81    -97.6    -2.01   47.0
랜덤 투표                    53.8       173.1    0.31    -55.4     0.97   53.6
완벽 예측 (상한선)          152529.9       236.0  646.23    -51.2  2976.64   64.6
Buy & Hold (벤치마크)       -41.4        59.4   -0.70    -32.2    -1.29   50.3


[CONSERVATIVE | 레버리지 2x]
                   연환산 수익률(%)  연환산 변동성(%)  Sharpe  최대낙폭(%)  Calmar  승률(%)
모멘텀 (추세 추종)             -23.0        65.6   -0.35    -38.1   -0.61   48.1
역추세                     -76.1        67.8   -1.12    -42.6   -1.79   49.7
랜덤 투표                    -7.5        56.1   -0.13    -23.7   -0.32   48.1
완벽 예측 (상한선)            1109.6        71.6   15.49    -23.7   46.80   63.5
Buy & Hold (벤치마크)       -41.4        59.4   -0.70    -32.2   -1.29   50.3



In [8]:
# ── 포트폴리오 비중 변화 (모멘텀 vs 역추세) — 숏 음수로 표시 ────────
coin_colors = {'BTC':'#F7931A','ETH':'#627EEA','SOL':'#9945FF','HYPE':'#00D4FF'}

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('포트폴리오 비중 변화 — 모멘텀 vs 역추세\n(양수=롱, 음수=숏)', fontsize=14, fontweight='bold')

compare_scenarios = ['momentum', 'contrarian']

for p_idx, profile_name in enumerate(FUND_PROFILES):
    for s_idx, scenario in enumerate(compare_scenarios):
        ax = axes[p_idx][s_idx]
        df = results[profile_name][scenario]
        x  = range(len(df))

        for coin in COINS:
            w = df[f'w_{coin}'].values
            # 롱(양수)과 숏(음수) 분리해서 스택
            long_w  = np.where(w > 0, w, 0)
            short_w = np.where(w < 0, w, 0)
            ax.fill_between(x, 0, long_w,  color=coin_colors[coin], alpha=0.7, label=f'{coin} 롱')
            ax.fill_between(x, 0, short_w, color=coin_colors[coin], alpha=0.3, hatch='//')

        ax.axhline(y=0, color='black', linewidth=0.8)
        ax.set_ylim(-80, 110)
        ax.set_ylabel('비중 (%)')
        ax.set_title(f'{profile_name.upper()} — {SCENARIO_LABELS[scenario]}')
        ax.text(0.01, 0.98, '위=롱 / 아래=숏(빗금)',
                transform=ax.transAxes, fontsize=8,
                verticalalignment='top', color='gray')
        if p_idx == 0 and s_idx == 1:
            handles = [plt.Rectangle((0,0),1,1, color=coin_colors[c], alpha=0.7)
                       for c in COINS]
            ax.legend(handles, COINS, loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('backtest_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ backtest_weights.png 저장')

✅ backtest_weights.png 저장


In [9]:
# ── 레버리지 민감도 분석 ──────────────────────────────────────────
# 모멘텀 시나리오 기준, 레버리지 1x~7x 변화에 따른 수익률/낙폭
LEVERAGES = [1, 2, 3, 5, 7]

lev_results = []
for lev in LEVERAGES:
    FUND_PROFILES['_test'] = {'T_CONVERGE': 10, 'MAX_WEIGHT': 70,
                              'FUND_LEVERAGE': lev, 'color': 'blue'}
    df = run_backtest('momentum', make_generator('momentum'), returns, '_test')
    m  = calc_metrics(df)
    m['레버리지'] = lev
    lev_results.append(m)

del FUND_PROFILES['_test']

lev_df = pd.DataFrame(lev_results).set_index('레버리지')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('레버리지 민감도 분석 (모멘텀 시나리오)', fontsize=14, fontweight='bold')

metrics_to_plot = [('연환산 수익률(%)', '#2ecc71', '연환산 수익률'),
                   ('최대낙폭(%)',      '#e74c3c', '최대 낙폭'),
                   ('Sharpe',           '#3498db', 'Sharpe Ratio')]

for ax, (col, color, title) in zip(axes, metrics_to_plot):
    ax.bar(LEVERAGES, lev_df[col], color=color, alpha=0.7, width=0.6)
    ax.plot(LEVERAGES, lev_df[col], 'o-', color=color, linewidth=2)
    ax.set_xlabel('레버리지 (x)')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xticks(LEVERAGES)

plt.tight_layout()
plt.savefig('leverage_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ leverage_sensitivity.png 저장')
print('\n레버리지별 수익률/리스크:')
print(lev_df[['연환산 수익률(%)', '최대낙폭(%)', 'Sharpe', '승률(%)']].to_string())

  ⚠️  momentum/_test: Day 46 청산 발생


✅ leverage_sensitivity.png 저장

레버리지별 수익률/리스크:
      연환산 수익률(%)  최대낙폭(%)  Sharpe  승률(%)
레버리지                                    
1           15.0    -25.7    0.34   49.7
2            9.3    -46.0    0.10   49.7
3          -16.0    -61.8   -0.12   49.7
5          -98.3    -83.4   -0.44   49.7
7         -168.8    -96.4   -0.54   49.7


## 5. 결론 체크리스트

백테스트 결과를 보고 확인할 것:

- [ ] 모멘텀이 역추세보다 일관되게 좋은가?
- [ ] 레버리지 몇 배에서 Sharpe가 꺾이는가? (그게 최적 레버리지)
- [ ] Aggressive가 Conservative보다 낙폭이 얼마나 크게 나오는가?
- [ ] 완벽 예측 대비 모멘텀의 수익률 비율 (알파 포착률)
- [ ] 청산 발생 레버리지 구간 확인 → 펀드 레버리지 상한 설정